# Feedforward Neural Network Breast Cancer Classifier

To establish a deep learning baseline for binary classification, we implemented a feedforward neural network using PyTorch on the breast cancer dataset from sklearn. The task is to predict whether a tumor is malignant or benign based on 30 real-valued features derived from digitized images of cell nuclei.

In [3]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Data Preprocessing

The dataset was split into training and test sets using an 80/20 stratified split to preserve class balance. Since the features vary significantly in scale, we applied standardization using StandardScaler, ensuring each feature has zero mean and unit variance. This is for stable optimization in neural networks.

In [5]:
# reproducibility
torch.manual_seed(42)
np.random.seed(42)

# load
data = load_breast_cancer()
X = data.data
y = data.target  # 0 = malignant, 1 = benign

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# standardize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
# conver to pytorch framework
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## Model Architecture

We used a simple multilayer perceptron (MLP) consisting of:

- Input layer (30 features)
- Hidden layer with 32 units and ReLU activation
- Hidden layer with 16 units and ReLU activation
- Output layer with 1 unit (logit)

This architecture provides a lightweight nonlinear model capable of capturing interactions between features while remaining computationally efficient.

In [8]:
# model architecture
class BreastCancerMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)

In [9]:
# define model
model = BreastCancerMLP(input_dim=X_train.shape[1])

# loss and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## Training Procedure

The model was trained using the binary cross-entropy loss with logits (BCEWithLogitsLoss), which is appropriate for binary classification tasks. We optimized the parameters using the Adam optimizer with a learning rate of $10^{−3}$. Training was performed for 100 epochs using mini-batches of size 32.

In [11]:
# training
num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss / len(train_loader):.4f}")

Epoch [10/100], Loss: 0.0960
Epoch [20/100], Loss: 0.0458
Epoch [30/100], Loss: 0.0315
Epoch [40/100], Loss: 0.0233
Epoch [50/100], Loss: 0.0164
Epoch [60/100], Loss: 0.0112
Epoch [70/100], Loss: 0.0081
Epoch [80/100], Loss: 0.0056
Epoch [90/100], Loss: 0.0039
Epoch [100/100], Loss: 0.0032


## Evaluation

Model performance was evaluated on the held-out test set using accuracy, along with a classification report and confusion matrix. Predictions were obtained by applying a sigmoid function to the output logits and thresholding at 0.5.

In [13]:
# evaulation
model.eval()
with torch.no_grad():
    test_logits = model(X_test_tensor)
    test_probs = torch.sigmoid(test_logits)
    test_preds = (test_probs >= 0.5).int().numpy().flatten()

y_true = y_test
acc = accuracy_score(y_true, test_preds)

print("\nTest Accuracy:", acc)
print("\nClassification Report:")
print(classification_report(y_true, test_preds, target_names=data.target_names))
print("Confusion Matrix:")
print(confusion_matrix(y_true, test_preds))


Test Accuracy: 0.956140350877193

Classification Report:
              precision    recall  f1-score   support

   malignant       0.91      0.98      0.94        42
      benign       0.99      0.94      0.96        72

    accuracy                           0.96       114
   macro avg       0.95      0.96      0.95       114
weighted avg       0.96      0.96      0.96       114

Confusion Matrix:
[[41  1]
 [ 4 68]]
